In [0]:
## Import Libraries
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from imblearn.under_sampling import RandomUnderSampler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix
import joblib
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature


In [0]:
url = r"C:\Users\Data Professor\Desktop\Hypertension Project\hypertension_dataset.csv"  ## csv_file

In [0]:
df = pd.read_csv("dataset.csv")

In [0]:
df.head()  ## frist 5 rows of dataset

In [0]:
df.shape  ## rows and coluns of the dataset

In [0]:
df.columns = df.columns.str.lower()  ## converting columns to lower case 
df.columns = df.columns.str.strip()  ## removing extra spaces in columns
    

In [0]:
df.info()  ## checking the information and structure of the dataset

In [0]:
df.duplicated().any()  ## checking for duplicated rows 

In [0]:
df.isna().any()  #3 checking for misiing values 

In [0]:
df.isna().sum()  ## identifying missing values 

In [0]:
df["medication"].value_counts()  ## medication count

In [0]:
df["medication"] = df["medication"].fillna("None")  ## handling missing values in medication column

In [0]:
df["medication"].isna().any()  ## confirming no null

In [0]:
df["has_hypertension"] = df["has_hypertension"].replace({"Yes":1, "No":0})  ## converting target to binary

In [0]:
df["has_hypertension"].value_counts()  ## target count to check for status balance

In [0]:
def train_data():
    df["medication"] = df["medication"].fillna("None")
    
    df["has_hypertension"] = df["has_hypertension"].replace({"Yes":1, "No":0})
    
    
    resampler = RandomUnderSampler()
    
    x = df.drop(["has_hypertension"], axis = 1)
    y = df["has_hypertension"]
    
    resampled_x, resampled_y = resampler.fit_resample(x,y)
    
    resampled_y.value_counts()
    
    
    x_train, x_test, y_train, y_test = train_test_split(resampled_x, resampled_y, test_size = .2, random_state = 42)
    
    return x_train, x_test, y_train, y_test

x_train, x_test, y_train, y_test = train_data()



def create_processor():
    num_cols = x_train.select_dtypes(include=["number"]).columns 
    cat_cols = x_train.select_dtypes(include=["object"]).columns
    
    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy = "median")),
        ("scaler", StandardScaler())
    ])
    
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy = "most_frequent")),
        ("scaler", OrdinalEncoder(handle_unknown = "use_encoded_value", unknown_value = -1))
    ])
    
    
    processor = ColumnTransformer(
        transformers=[
            ("num_pipe", num_pipe, num_cols),
            ("cat_pipe", cat_pipe, cat_cols)
        ], 
        remainder = "passthrough"
    )
    return processor

processor = create_processor()



def check_models():
    Models = {
        "Linear Model" : LogisticRegression(),
        "Tree Model" : DecisionTreeClassifier(),
        "Random Forest Model" : RandomForestClassifier(),
        "Vector Model" : SVC(),
        "Neighbors Model" : KNeighborsClassifier(),
        "XGBoost" : XGBClassifier()
    }
    
    for name, models in Models.items():
        model = Pipeline([
            ("transformer", processor),
            ("model", models)
        ]).fit(x_train, y_train)
        prediction = model.predict(x_test)
    
        report = classification_report(y_test, prediction)
        matrix = confusion_matrix(y_test, prediction)
        print(f"The Classification Report for {name}:")
        print(report)
        print()
        print(f"The Confusion Matrix for {name}:")
        print(matrix)
        print("-"*100)


check_models()
    
    

In [0]:
## xgboost model 
def train_xgboost():
    # Start MLflow run
    with mlflow.start_run(run_name = 'hypertension xgboost model'):
        # Train the XGBoost model
        xgboost_model = Pipeline([
            ("transformer", processor),  # Assuming 'processor' is your pre-processing pipeline
            ("xgboost_model", XGBClassifier())
        ]).fit(x_train, y_train)

        # Log model signature (required for Unity Catalog)
        signature = infer_signature(x_train, xgboost_model.predict(x_train))

        # Log the model with signature
        mlflow.sklearn.log_model(
            sk_model=xgboost_model,
            artifact_path="hypertension_xgboost_model",
            registered_model_name="Hypertension_XGBoost_Model",  # Name for model registry
            signature=signature  # Include signature for Unity Catalog
        )

        # Log metrics (accuracy)
        accuracy = xgboost_model.score(x_test, y_test)
        mlflow.log_metric("accuracy", accuracy)
        print(f"Model Accuracy: {accuracy:.4f}")
        
        # Feature importance plot
        features = x_train.columns
        importance = xgboost_model.named_steps["xgboost_model"].feature_importances_

        importance_df = pd.DataFrame({
            "Features": features,
            "Importance Score": importance
        })

        # Save and log feature importance plot
        fig, ax = plt.subplots()
        importance_df.plot(x="Features", y="Importance Score", kind="barh", ax=ax)
        plt.title("Features Vs Importance Score")
        plt.savefig("feature_importance_plot.png")
        mlflow.log_artifact("feature_importance_plot.png")  # Log plot as artifact
        plt.close()

        # Log top 5 important features plot
        ranked_importance = importance_df.sort_values(by="Importance Score", ascending=False).reset_index(drop=True).head().round(2)
        fig, ax = plt.subplots()
        ranked_importance.plot(x="Features", y="Importance Score", kind="barh", ax=ax)
        plt.title("Top 5 Feature Importance By Score")
        plt.savefig("top_5_feature_importance_plot.png")
        mlflow.log_artifact("top_5_feature_importance_plot.png")  # Log plot as artifact
        plt.close()

        # End the MLflow run
        mlflow.end_run()

train_xgboost()